In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"


In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

# 1. Define the Graph (0 connected to 1, 1 connected to 2)
edge_index = torch.tensor([[0, 1, 1, 2],
                           [1, 0, 2, 1]], dtype=torch.long)

# 3 nodes, each with 2 features
x = torch.tensor([[2.0, 1.0], 
                  [5.0, 3.0], 
                  [1.0, 8.0]], dtype=torch.float)

# Target labels for the 3 nodes
y = torch.tensor([0, 1, 0], dtype=torch.long)

# Simple mask to train on all nodes for this toy example
train_mask = torch.tensor([True, True, True], dtype=torch.bool)

data = Data(x=x, edge_index=edge_index, y=y, train_mask=train_mask)

# 2. Define the GNN Architecture
class SimpleGCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 2 input features, 16 hidden channels, 2 output classes
        self.conv1 = GCNConv(2, 16)
        self.conv2 = GCNConv(16, 2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# 3. Train the Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleGCN().to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

model.train()
for epoch in range(1000):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:03d}, Loss: {loss.item():.4f}')


Epoch 020, Loss: 0.6521
Epoch 040, Loss: 0.6189
Epoch 060, Loss: 0.5843
Epoch 080, Loss: 0.5440
Epoch 100, Loss: 0.4995
Epoch 120, Loss: 0.4497
Epoch 140, Loss: 0.4008
Epoch 160, Loss: 0.3494
Epoch 180, Loss: 0.3028
Epoch 200, Loss: 0.2603
Epoch 220, Loss: 0.2227
Epoch 240, Loss: 0.1923
Epoch 260, Loss: 0.1668
Epoch 280, Loss: 0.1436
Epoch 300, Loss: 0.1248
Epoch 320, Loss: 0.1099
Epoch 340, Loss: 0.0965
Epoch 360, Loss: 0.0856
Epoch 380, Loss: 0.0763
Epoch 400, Loss: 0.0686
Epoch 420, Loss: 0.0621
Epoch 440, Loss: 0.0563
Epoch 460, Loss: 0.0513
Epoch 480, Loss: 0.0473
Epoch 500, Loss: 0.0442
Epoch 520, Loss: 0.0408
Epoch 540, Loss: 0.0382
Epoch 560, Loss: 0.0357
Epoch 580, Loss: 0.0336
Epoch 600, Loss: 0.0317
Epoch 620, Loss: 0.0303
Epoch 640, Loss: 0.0288
Epoch 660, Loss: 0.0276
Epoch 680, Loss: 0.0264
Epoch 700, Loss: 0.0254
Epoch 720, Loss: 0.0245
Epoch 740, Loss: 0.0237
Epoch 760, Loss: 0.0229
Epoch 780, Loss: 0.0224
Epoch 800, Loss: 0.0217
Epoch 820, Loss: 0.0211
Epoch 840, Loss:

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import EdgeConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
# ==========================================
# GNN COMPONENTS (Backbone & Tasks)
# ==========================================
class IceCubeGNNBackbone(nn.Module):
    def __init__(self, in_feats=5, hidden_feats=64):
        super().__init__()
        nn_layer = nn.Sequential(
            nn.Linear(in_feats * 2, hidden_feats),
            nn.ReLU(),
            nn.Linear(hidden_feats, hidden_feats)
        )
        self.conv1 = EdgeConv(nn_layer, aggr='max')
        
    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        return torch.relu(h)

class SSLPretrainingModel(nn.Module):
    def __init__(self, backbone, hidden_feats=64):
        super().__init__()
        self.backbone = backbone
        self.ssl_head = nn.Sequential(
            nn.Linear(hidden_feats, 32),
            nn.ReLU(),
            nn.Linear(32, 1) 
        )
        
    def forward(self, x, edge_index):
        latent_nodes = self.backbone(x, edge_index)
        return self.ssl_head(latent_nodes)

class LabeledFlavorModel(nn.Module):
    def __init__(self, pretrained_backbone, hidden_feats=64, num_classes=3):
        super().__init__()
        self.backbone = pretrained_backbone
        self.flavor_head = nn.Sequential(
            nn.Linear(hidden_feats, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x, edge_index, batch_mapping):
        latent_nodes = self.backbone(x, edge_index)
        graph_embedding = global_mean_pool(latent_nodes, batch_mapping)
        return self.flavor_head(graph_embedding)


# ==========================================
# DATA FABRICATION (Mocking IceCube Events)
# ==========================================
def generate_mock_icecube_data(num_events=50):
    """Generates synthetic IceCube collision data graphs."""
    dataset = []
    for i in range(num_events):
        # A variable number of DOM sensors light up per event (e.g., between 10 and 30)
        num_hits = torch.randint(10, 30, (1,)).item()
        
        # Node features: [X, Y, Z, Charge, Arrival_Time]
        pos = torch.randn(num_hits, 3) * 500.0     # Simulated coordinates deep in ice
        charge = torch.rand(num_hits, 1) * 10.0     # Photon counts captured
        time = torch.sort(torch.rand(num_hits, 1) * 2000.0)[0] # Chronological timestamps
        
        x = torch.cat([pos, charge, time], dim=-1)
        
        # Generate simple edge connections (connecting subsequent light pulses)
        src = torch.arange(0, num_hits - 1)
        dst = torch.arange(1, num_hits)
        edge_index = torch.stack([torch.cat([src, dst]), torch.cat([dst, src])], dim=0)
        
        # Ground-truth Flavor Label: 0=Electron, 1=Muon, 2=Tau
        flavor_label = torch.randint(0, 3, (1,)).item()
        
        # Construct the PyG Data object
        data = Data(x=x, edge_index=edge_index, y=torch.tensor([flavor_label], dtype=torch.long))
        dataset.append(data)
        
    return dataset


# ==========================================
# PIPELINE EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Setup Data Loaders
    print("Step 1: Fabricating synthetic IceCube events...")
    mock_events = generate_mock_icecube_data(num_events=100)
    
    # In practice, your SSL data could contain thousands of unlabeled events
    ssl_loader = DataLoader(mock_events[:30], batch_size=4, shuffle=True)
    fine_tune_loader = DataLoader(mock_events[30:], batch_size=4, shuffle=False)
    
    # 2. Instantiate Shared Backbone
    backbone = IceCubeGNNBackbone(in_feats=5, hidden_feats=64)
    
    # 3. Execution: Stage 1 - Self-Supervised Pretraining
    print("\nStep 2: Commencing Self-Supervised Pretraining (SSL)...")
    ssl_model = SSLPretrainingModel(backbone)
    ssl_optimizer = torch.optim.Adam(ssl_model.parameters(), lr=0.01)
    ssl_criterion = nn.MSELoss()
    
    for epoch in range(100): # Small epochs for testing
        total_loss = 0
        for batch in ssl_loader:
            ssl_optimizer.zero_grad()
            
            # Masking step: Clones the true arrival times and zeros out the input feature
            true_times = batch.x[:, 4].clone()
            masked_x = batch.x.clone()
            masked_x[:, 4] = 0.0 
            
            predictions = ssl_model(masked_x, batch.edge_index).squeeze(-1)
            loss = ssl_criterion(predictions, true_times)
            
            loss.backward()
            ssl_optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/100 | SSL Reconstruct Loss: {total_loss/len(ssl_loader):.4f}")
        
    # 4. Execution: Stage 2 - Fine-Tuning Flavor Head
    print("\nStep 3: Transferring Backbone & Fine-Tuning the Flavor Head...")
    # Passing the exact 'backbone' object carries over the learned weights
    classifier = LabeledFlavorModel(pretrained_backbone=backbone, hidden_feats=64, num_classes=3)
    clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001) # Slower LR
    clf_criterion = nn.CrossEntropyLoss()
    
    for epoch in range(100):
        total_loss = 0
        correct = 0
        total_nodes = 0
        
        for batch in fine_tune_loader:
            clf_optimizer.zero_grad()
            
            outputs = classifier(batch.x, batch.edge_index, batch.batch)
            loss = clf_criterion(outputs, batch.y)
            
            loss.backward()
            clf_optimizer.step()
            total_loss += loss.item()
            
            # Tracking metrics
            predictions = outputs.argmax(dim=-1)
            correct += (predictions == batch.y).sum().item()
            
        acc = (correct / len(fine_tune_loader.dataset)) * 100
        print(f"Epoch {epoch+1}/100 | Fine-Tuning Loss: {total_loss/len(fine_tune_loader):.4f} | Accuracy: {acc:.1f}%")


Step 1: Fabricating synthetic IceCube events...

Step 2: Commencing Self-Supervised Pretraining (SSL)...
Epoch 1/10 | SSL Reconstruct Loss: 860679.9180
Epoch 2/10 | SSL Reconstruct Loss: 424962.3945
Epoch 3/10 | SSL Reconstruct Loss: 405645.1289
Epoch 4/10 | SSL Reconstruct Loss: 384127.6797
Epoch 5/10 | SSL Reconstruct Loss: 366967.0820
Epoch 6/10 | SSL Reconstruct Loss: 360697.8047
Epoch 7/10 | SSL Reconstruct Loss: 358057.8867
Epoch 8/10 | SSL Reconstruct Loss: 356498.1055
Epoch 9/10 | SSL Reconstruct Loss: 345108.9531
Epoch 10/10 | SSL Reconstruct Loss: 337809.2617
Epoch 11/10 | SSL Reconstruct Loss: 339801.5820
Epoch 12/10 | SSL Reconstruct Loss: 340041.7344
Epoch 13/10 | SSL Reconstruct Loss: 352545.8867
Epoch 14/10 | SSL Reconstruct Loss: 356064.7070
Epoch 15/10 | SSL Reconstruct Loss: 338453.8594
Epoch 16/10 | SSL Reconstruct Loss: 335228.4180
Epoch 17/10 | SSL Reconstruct Loss: 327226.3945
Epoch 18/10 | SSL Reconstruct Loss: 325546.8125
Epoch 19/10 | SSL Reconstruct Loss: 3220